# EXPERIMENTING

In [1]:
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np
import pthelperfunctions as helper

pd.set_option('display.max_rows', None)

#path_mob = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/mobilization_output"
#path_ptot = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/ot_pt_output"

📢 ClifOrchestrator initialized


In [2]:
d_items_df = helper.load_data('mimic','d_items',folder='icu', type='csv')
d_items_df.head()

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,220001,Problem List,Problem List,chartevents,General,NaN,Text,NaN,NaN
1,220003,ICU Admission date,ICU Admission date,datetimeevents,ADT,NaN,Date and time,NaN,NaN
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,NaN,NaN
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,NaN,NaN


In [3]:
d_items_df['category'].value_counts()

category
Skin - Impairment                     412
Access Lines - Invasive               372
Respiratory                           170
Labs                                  161
Medications                           139
Care Plans                            137
Skin - Incisions                      115
Nutrition - Enteral                   113
Ingredients - general (Not In Use)    109
Pain/Sedation                          97
Treatments                             93
Neurological                           88
Restraint/Support Systems              88
MD Progress Note                       85
Fluids - Other (Not In Use)            84
Access Lines - Peripheral              75
GI/GU                                  73
PA Line Insertion                      63
Bronchoscopy                           61
Scores - APACHE IV (2)                 60
Antibiotics                            55
Intubation                             55
CVL Insertion                          53
ECMO                     

In [4]:
'''
Picked categories which appear to be relevant from above.
'''
categories_to_get = ['Care Plans','Treatments','Generic Proc Note','3-Significant Events','4-Procedures','Case Management']
filtered_items = d_items_df[d_items_df['category']==categories_to_get[5]]
print(filtered_items.shape[0])
filtered_items

2


,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
1286,225411,Patient on vent,Patient on vent,chartevents,Case Management,NaN,Text,NaN,NaN
1287,225414,Home without services,Home without services,chartevents,Case Management,NaN,Checkbox,NaN,NaN


In [5]:
'''
Check all D-ITEMS manually by iterating the cell above and hand select the ones we want.
'''
items_to_get = [229344,229347,229349,229350, #Impoired mobility, Care Plans, chartevents
                224084,224086, 229319, 229321, 229633, 229634, 229635, 229636, 229637, 229638, 229681, #Activity/HLM/AMPAC, Treatments, chartevents
                225474, #Fall, 3-Significant Events, procedureevents
                225414] #Home without service, Case Management, chartevents
filtered_items = d_items_df[d_items_df['itemid'].isin(items_to_get)]

In [6]:
#Load block data from our project output.
path = os.path.join(helper.path_out, "block_compiled_5.parquet")
block_final = pd.read_parquet(path)

#Separate procedureevents and chartevents
items_chart_df = d_items_df[(d_items_df['linksto']=='chartevents') & d_items_df['itemid'].isin(items_to_get)]
items_proc_df = d_items_df[(d_items_df['linksto']=='procedureevents') & d_items_df['itemid'].isin(items_to_get)]

In [ ]:
#Load mimic chart events
chart_df = helper.load_data("mimic","chartevents", folder='icu')
chart_df.rename(columns={'subject_id': 'patient_id','hadm_id':'hospitalization_id'}, inplace=True)

#Filter for our cohort and needs
chart_mask = \
        chart_df['hospitalization_id'].isin(block_final['hospitalization_id'].astype(int)) &\
        chart_df['itemid'].isin(items_chart_df['itemid']) &\
        chart_df['value'].notna()

filtered_chart_df = chart_df[chart_mask]

#Merge with D_items Labels
filtered_chart_df = filtered_chart_df.merge(
    items_chart_df[['itemid','label']],
    on='itemid',
    how='left')

In [8]:
#Merge with cohort data
block_final['hospitalization_id'] = block_final['hospitalization_id'].astype(int)
filtered_chart_df = filtered_chart_df.merge(
    block_final[['encounter_block','hospitalization_id','pt_post48_IMV']],
    on='hospitalization_id',
    how='left')

#Pivot Table
pivot_chart_table = filtered_chart_df.pivot_table(
    values='encounter_block',
    index=['label','itemid'],
    columns='pt_post48_IMV',
    aggfunc='nunique',
    fill_value=0
)
pivot_chart_table = pivot_chart_table.reset_index()

#Print DF
print(pivot_chart_table)

pt_post48_IMV                                      label  itemid  False  True
0                                               Activity  224084  12306  2160
1                           Activity / Mobility (JH-HLM)  229319    320    52
2                           Activity / Mobility (JH-HLM)  229321  14731  2551
3                                     Activity Tolerance  224086  27246  4756
4                                Basic Mobility (AM-PAC)  229633   1581   330
5                                Daily Activity (AM-PAC)  229634    184    44
6                     Discharge Recommendations (AM-PAC)  229638   1643   337
7                 Impaired Mobility  NCP - Interventions  229347   1385   268
8              Impaired Mobility NCP - Expected outcomes  229344   1375   270
9                   Impaired Mobility NCP - Plan revised  229350   1376   269
10              Impaired Mobility NCP - Problem resolved  229349   1358   266
11                      Indwelling Urinary Catheter Care  229681

In [10]:
#Load mimic proc events
proc_df = helper.load_data("mimic","procedureevents", folder='icu', type='csv.gz')
proc_df.rename(columns={'subject_id': 'patient_id','hadm_id':'hospitalization_id'}, inplace=True)

#Filter for our cohort and needs
proc_mask = \
        proc_df['hospitalization_id'].isin(block_final['hospitalization_id'].astype(int)) &\
        proc_df['itemid'].isin(items_proc_df['itemid']) &\
        proc_df['value'].notna()

filtered_proc_df = proc_df[proc_mask]

#Merge with D_items Labels
filtered_proc_df = filtered_proc_df.merge(
    items_proc_df[['itemid','label']],
    on='itemid',
    how='left')

#Merge with cohort data
filtered_proc_df = filtered_proc_df.merge(
    block_final[['encounter_block','hospitalization_id','pt_post48_IMV']],
    on='hospitalization_id',
    how='left')

#Pivot Table
pivot_proc_table = filtered_proc_df.pivot_table(
    values='encounter_block',
    index=['label','itemid'],
    columns='pt_post48_IMV',
    aggfunc='nunique',
    fill_value=0
)
pivot_proc_table = pivot_proc_table.reset_index()

#Print DF
print(pivot_proc_table)

pt_post48_IMV label  itemid  False  True
0              Fall  225474     29     8


Now look for the CLIF Procedure Table

In [1]:
import pandas as pd
import numpy as np
import os
import pthelperfunctions as helper

path = os.path.join(helper.path_out, "block_compiled_5.parquet")
df = pd.read_parquet(path)
proc_df = helper.load_clif_table('patient_procedures', hosp_list=df['hospitalization_id'].tolist())

codes_list = pd.read_csv(os.path.join("..","config","therapy_codes.csv"))
codes_list['code'] = codes_list['code'].astype(str)
codes_list.rename(columns={'code':'procedure_code'}, inplace=True)
cpt_df = pd.merge(
    proc_df,
    codes_list,
    on='procedure_code',
    how='inner'
)
cpt_df.head(10)

📢 ClifOrchestrator initialized
📢 Initialized patient_procedures table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/patient_procedures_schema.yaml
📢 Loaded outlier configuration
📢 Initialized patient_procedures table
📢 Data directory: /nfs/roberts/project/pi_sj692/shared/mimicclif
📢 File type: parquet
📢 Timezone: America/New_York
📢 Output directory: /nfs/roberts/project/pi_sj692/shared/PT_consults/output
📢 Loaded schema from /nfs/roberts/project/pi_sj692/gcr22/clifenv/lib/python3.12/site-packages/clifpy/schemas/patient_procedures_schema.yaml
📢 Loaded outlier configuration
Using CLIF standard outlier ranges

No outlier configuration found for table: patient_procedures


,hospitalization_id,billing_provider_id,performing_provider_id,procedure_code,procedure_code_format,procedure_billed_dttm,description,pt_binary
0,23231225,<NA>,<NA>,97605,CPT,2195-01-21 00:00:00-05:00,Neg press wound tx < 50 cm,0
1,22284188,<NA>,<NA>,97597,CPT,2143-07-14 00:00:00-05:00,RMVL devital tis 20cm/<,0
2,24690214,<NA>,<NA>,97605,CPT,2162-10-31 00:00:00-05:00,Neg press wound tx < 50 cm,0
3,28150037,<NA>,<NA>,97597,CPT,2181-09-30 00:00:00-05:00,RMVL devital tis 20cm/<,0
4,26370765,<NA>,<NA>,97605,CPT,2137-04-14 00:00:00-05:00,Neg press wound tx < 50 cm,0
5,26266189,<NA>,<NA>,97597,CPT,2185-01-08 00:00:00-05:00,RMVL devital tis 20cm/<,0
6,27331302,<NA>,<NA>,97597,CPT,2185-12-12 00:00:00-05:00,RMVL devital tis 20cm/<,0
7,29641685,<NA>,<NA>,97597,CPT,2169-02-27 00:00:00-05:00,RMVL devital tis 20cm/<,0
8,21160436,<NA>,<NA>,97597,CPT,2115-03-25 00:00:00-05:00,RMVL devital tis 20cm/<,0
9,25314475,<NA>,<NA>,97597,CPT,2179-05-22 00:00:00-05:00,RMVL devital tis 20cm/<,0


In [7]:
proc_df['procedure_code_format'].value_counts()

procedure_code_format
ICD9        469209
ICD10PCS    390446
CPT         116032
HCPCS        70042
Name: count, dtype: int64